# nf-core/rnaseq Practice Notebook

This notebook walks through the **nf-core/rnaseq** pipeline - a production-grade, complex workflow for learning advanced Nextflow concepts.

## Why RNA-seq After Demo?
- 🎓 **Production-grade** - Real-world complexity, not a toy example
- 🔀 **Subworkflows** - Modular pipeline sections (new concept!)
- ⚙️ **Conditional Logic** - Skip flags and branching paths
- 🛠️ **Tool Choice** - Multiple quantification methods (Salmon, STAR, RSEM)
- 📊 **Complex DAG** - See how large workflows are structured

## What's NEW vs. nf-core/demo?

| Concept | Demo | RNA-seq |
|---------|------|--------|
| Processes | 2-3 simple | 30+ complex |
| Subworkflows | None | Multiple nested |
| Conditional execution | No | Yes (skip flags) |
| Tool alternatives | No | Yes (alignment methods) |
| Pipeline branches | Linear | Multiple paths |
| Real-world complexity | Tutorial | Production |

## Learning Goals
1. **Subworkflows** - How to compose workflows from smaller pieces
2. **Conditional execution** - `--skip_*` parameters and branching
3. **Tool alternatives** - Running different quantifiers
4. **DAG visualization** - Understanding complex workflow graphs
5. **Module organization** - How nf-core structures large pipelines

---
## Setup (One-time)

RNA-seq is already installed on your system! Just verify.

### 1. Verify Installation

In [1]:
# Check if rnaseq is already pulled
!nextflow list | grep rnaseq

nf-core/rnaseq


In [4]:
# If not found above, pull it:
!nextflow pull nf-core/rnaseq

Checking nf-core/rnaseq ...
 Already-up-to-date - revision: e7ca46272c [master]


### 2. Check Pipeline Info

In [5]:
# View pipeline information
!nextflow info nf-core/rnaseq

 project name: nf-core/rnaseq
 repository  : https://github.com/nf-core/rnaseq
 local path  : /home/sagemaker-user/.nextflow/assets/.repos/nf-core/rnaseq
 main script : main.nf
 description : RNA sequencing analysis pipeline for gene/isoform quantification and extensive quality control.
 revisions   : 
   TEMPLATE
   address-famke-1838-feedback
   admonitions
   arm-CI
   bisect-salmon-bam-fasta-mismatch
   bump-version-3.23.0dev
   bump-version-3.26.0dev
   bump-version-3.27.0dev
   bump/post-release-3.22.1
   change_testing_logic
   ci/arm-docker-only
   claude/blissful-goodall-a315fe
   codex/parabricks-star-index-precedence
   container-configs
   contaminant-screening-input
   copilot/explore-nextflow-topics
   copilot/update-workflow-output-syntax
   dev
   docs/star-rsem-extra-args
   drop-legacy-star
   dupradar-fusion
   edmundmiller/fq-lint
   exclude_star_rsem_pca_from_snaps
   feat-multiqc-ai-summaries
   feat/implement-ci-nf-tests
   feat/metro-process-map
   feat/seq-plat

### 3. Create Working Directory

In [6]:
import os

# Create working directory
work_dir = "/tmp/nextflow_rnaseq_practice"
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

print(f"Working directory: {os.getcwd()}")

Working directory: /tmp/nextflow_rnaseq_practice


---
## Part 1: Full Pipeline Run

First, let's run the complete pipeline to see all the pieces working together.

### 4. Run Full RNA-seq Pipeline with Test Data

**This will take ~30-45 minutes** - it's doing real alignment and quantification!  
Uses tiny test data but runs the full workflow.

In [ ]:
%%bash

# Run the full pipeline with all reports
# Use .svg for DAG to display directly in notebook
nextflow run nf-core/rnaseq \
    -profile test,conda \
    --outdir results_rnaseq_full \
    -with-report report_full.html \
    -with-timeline timeline_full.html \
    -with-trace trace_full.txt \
    -with-dag dag_full.svg

**What just happened?**
- Quality control (FastQC)
- Adapter trimming (if needed)
- Alignment to reference genome (STAR)
- Quantification (Salmon + featureCounts)
- Quality metrics (RSeQC, Qualimap)
- MultiQC report aggregation

**This is a REAL production pipeline!**

### 5. Explore the Output Structure

In [10]:
# See the complex output directory structure
!tree -L 2 -d results_rnaseq_full

results_rnaseq_full
├── bbsplit
├── fastqc
│   ├── filtered
│   ├── raw
│   └── trim
├── fq_lint
│   ├── bbsplit
│   ├── raw
│   └── trimmed
├── multiqc
│   └── star_salmon
├── pipeline_info
├── salmon
│   ├── RAP1_IAA_30M_REP1
│   ├── RAP1_UNINDUCED_REP1
│   ├── RAP1_UNINDUCED_REP2
│   ├── WT_REP1
│   ├── WT_REP2
│   └── deseq2_qc
├── star_salmon
│   ├── RAP1_IAA_30M_REP1
│   ├── RAP1_UNINDUCED_REP1
│   ├── RAP1_UNINDUCED_REP2
│   ├── WT_REP1
│   ├── WT_REP2
│   ├── bigwig
│   ├── deseq2_qc
│   ├── dupradar
│   ├── featurecounts
│   ├── log
│   ├── picard_metrics
│   ├── qualimap
│   ├── rseqc
│   ├── samtools_stats
│   └── stringtie
└── trimgalore

37 directories


**Notice the organization:**
- `fastqc/` - QC reports
- `star_salmon/` - Alignment and quantification
- `multiqc/` - Aggregated report
- `pipeline_info/` - Execution metadata

---
## Part 2: Understanding the DAG (Workflow Graph)

### 6. View the DAG

Download `dag_full.html` to see the complete workflow graph.  
**This shows how complex workflows branch and merge!**

In [ ]:
# Check DAG file was created
!ls -lh dag_full.svg

In [ ]:
# Display the DAG directly in the notebook
from IPython.display import SVG, display

print("🔀 Full Pipeline DAG (scroll to zoom):")
display(SVG(filename='dag_full.svg'))

**In the DAG you'll see:**
- Parallel FastQC on all samples
- STAR index creation (once)
- Per-sample alignment
- Multiple quantification methods
- Convergence to MultiQC

**Compare this to demo's simple linear flow!**

---
## Part 3: Conditional Execution (Skip Flags)

Now let's explore how to control which parts of the pipeline run.

### 7. Run with Salmon Only (Skip Alignment)

This demonstrates **conditional workflow paths** - a key difference from demo.

In [ ]:
%%bash

# Run only pseudo-alignment with Salmon, skip full alignment
nextflow run nf-core/rnaseq \
    -profile test,conda \
    --pseudo_aligner salmon \
    --skip_alignment \
    --outdir results_salmon_only \
    -with-dag dag_salmon.svg \
    -resume

**What changed?**
- No STAR alignment (skipped)
- No BAM files
- Only Salmon quantification
- Much faster!

**Compare `dag_salmon.html` to `dag_full.html`** - see the branches that were skipped!

In [ ]:
# Display the Salmon-only DAG to compare
from IPython.display import SVG, display

print("🐟 Salmon-only Pipeline DAG (compare to full pipeline above):")
display(SVG(filename='dag_salmon.svg'))

### 8. Compare Execution Times

In [15]:
# View execution history
!nextflow log -f "run_name,duration,status"

TIMESTAMP          	DURATION	RUN NAME         	STATUS	REVISION ID	SESSION ID                          	COMMAND                                                                                                                                                                                        
2026-06-29 20:03:53	12m 56s 	happy_cori       	OK    	e7ca46272c 	a6a7e1ce-2fdc-4539-a7d6-13f57c164708	nextflow run nf-core/rnaseq -profile test,conda --outdir results_rnaseq_full -with-report report_full.html -with-timeline timeline_full.html -with-trace trace_full.txt -with-dag dag_full.html
2026-06-29 20:31:13	48.3s   	stupefied_maxwell	ERR   	e7ca46272c 	a6a7e1ce-2fdc-4539-a7d6-13f57c164708	nextflow run nf-core/rnaseq -profile test,conda --outdir results_rnaseq_full -with-dag dag_full.svg -resume -dump-channels                                                                     


**Notice:** Salmon-only run was much faster because it skipped alignment!

---
## Part 4: Exploring Subworkflows

**Subworkflows** are the key organizational pattern for complex pipelines.

### 9. List Subworkflows

In [16]:
# See all subworkflows in the pipeline
!ls -1 ~/.nextflow/assets/nf-core/rnaseq/subworkflows/local/

align_star
prepare_genome
quantify_pseudo
quantify_rsem


**These are composable workflow chunks:**
- `align_star/` - Full STAR alignment workflow
- `quantify_salmon/` - Salmon quantification workflow
- `bam_*` - BAM file processing workflows

Each subworkflow contains multiple processes working together!

### 10. Study a Subworkflow Example

In [17]:
# View the Salmon quantification subworkflow
!cat ~/.nextflow/assets/nf-core/rnaseq/subworkflows/local/quantify_salmon.nf

cat: /home/sagemaker-user/.nextflow/assets/nf-core/rnaseq/subworkflows/local/quantify_salmon.nf: No such file or directory


**Key observations:**
- It calls multiple modules (SALMON_INDEX, SALMON_QUANT, etc.)
- Has its own input/output declarations
- Can be reused in different pipelines
- Encapsulates complex logic

**Demo didn't have this** - it was a flat list of processes!

---
## Part 5: Module Organization

### 11. Count the Modules

In [18]:
# See how many modules this pipeline uses
!find ~/.nextflow/assets/nf-core/rnaseq/modules/nf-core -name "main.nf" | wc -l

45


In [19]:
# List some key modules
!find ~/.nextflow/assets/nf-core/rnaseq/modules/nf-core -type d -maxdepth 1 | head -20

find: warning: you have specified the global option -maxdepth after the argument -type, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.
/home/sagemaker-user/.nextflow/assets/nf-core/rnaseq/modules/nf-core
/home/sagemaker-user/.nextflow/assets/nf-core/rnaseq/modules/nf-core/bbmap
/home/sagemaker-user/.nextflow/assets/nf-core/rnaseq/modules/nf-core/cat
/home/sagemaker-user/.nextflow/assets/nf-core/rnaseq/modules/nf-core/fastp
/home/sagemaker-user/.nextflow/assets/nf-core/rnaseq/modules/nf-core/fastqc
/home/sagemaker-user/.nextflow/assets/nf-core/rnaseq/modules/nf-core/fq
/home/sagemaker-user/.nextflow/assets/nf-core/rnaseq/modules/nf-core/gffread
/home/sagemaker-user/.nextflow/assets/nf-core/rnaseq/modules/nf-core/gunzip
/home/sagemaker-user/.nextflow/assets/nf-core/rnaseq/modules/nf-core/hisat2
/home/sagemaker-user/.nextflow/assets/nf-core/rnaseq/modules/n

**Compare to demo:** RNA-seq has 30+ modules vs demo's 2-3!

### 12. Study a Complex Module

In [20]:
# Look at the STAR alignment module
!cat ~/.nextflow/assets/nf-core/rnaseq/modules/nf-core/star/align/main.nf

process STAR_ALIGN {
    tag "$meta.id"
    label 'process_high'

    conda "bioconda::star=2.7.10a bioconda::samtools=1.16.1 conda-forge::gawk=5.1.0"
    container "${ workflow.containerEngine == 'singularity' && !task.ext.singularity_pull_docker_container ?
        'https://depot.galaxyproject.org/singularity/mulled-v2-1fa26d1ce03c295fe2fdcf85831a92fbcbd7e8c2:1df389393721fc66f3fd8778ad938ac711951107-0' :
        'biocontainers/mulled-v2-1fa26d1ce03c295fe2fdcf85831a92fbcbd7e8c2:1df389393721fc66f3fd8778ad938ac711951107-0' }"

    input:
    tuple val(meta), path(reads, stageAs: "input*/*")
    tuple val(meta2), path(index)
    tuple val(meta3), path(gtf)
    val star_ignore_sjdbgtf
    val seq_platform
    val seq_center

    output:
    tuple val(meta), path('*Log.final.out')   , emit: log_final
    tuple val(meta), path('*Log.out')         , emit: log_out
    tuple val(meta), path('*Log.progress.out'), emit: log_progress
    path  "versions.yml"                      , emit: versions


**Notice the complexity:**
- Multiple input channels
- Conditional outputs
- Resource directives (CPUs, memory)
- Error handling
- Container/conda definitions

---
## Part 6: Understanding the Main Workflow

### 13. View Main Workflow Structure

In [21]:
# See the top-level workflow
!cat ~/.nextflow/assets/nf-core/rnaseq/workflows/rnaseq.nf | head -100

/*
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    PRINT PARAMS SUMMARY
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
*/

include { paramsSummaryLog; paramsSummaryMap; fromSamplesheet } from 'plugin/nf-validation'

def logo = NfcoreTemplate.logo(workflow, params.monochrome_logs)
def citation = '\n' + WorkflowMain.citation(workflow) + '\n'
def summary_params = paramsSummaryMap(workflow)

// Print parameter summary log to screen
log.info logo + paramsSummaryLog(workflow) + citation

/*
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    VALIDATE INPUTS
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
*/

WorkflowRnaseq.initialise(params, log)

// Check rRNA databases for sortmerna
if (params.remove_ribo_rna) {
    ch_ribo_db = file(params.ribo_database_manifest)
    if (ch_ribo_db.isEmpty()) {exit 1, "File provided with -

**Key patterns:**
1. Input validation
2. Reference preparation
3. Conditional subworkflow inclusion (`if` statements)
4. Channel merging for MultiQC
5. Output organization

### 14. Find Conditional Logic

In [22]:
# Search for skip logic in the workflow
!grep -n "skip" ~/.nextflow/assets/nf-core/rnaseq/workflows/rnaseq.nf | head -20

31:if (!params.skip_bbsplit && !params.bbsplit_index && params.bbsplit_fasta_list) {
38:if (!params.skip_bbsplit) { prepareToolIndices << 'bbsplit' }
39:if (!params.skip_alignment) { prepareToolIndices << params.aligner }
40:if (!params.skip_pseudo_alignment && params.pseudo_aligner) { prepareToolIndices << params.pseudo_aligner }
46:        !params.skip_alignment && params.aligner
50:        !params.skip_pseudo_alignment && params.pseudo_aligner
57:        // Condition 4: --skip_gtf_filter is not provided
58:        !params.skip_gtf_filter
203:    if (!params.skip_alignment && !params.bam_csi_index) {
258:            params.skip_fastqc || params.skip_qc,
260:            params.skip_umi_extract,
261:            params.skip_trimming,
279:            params.skip_fastqc || params.skip_qc,
281:            params.skip_umi_extract,
283:            params.skip_trimming,
320:    if (!params.skip_bbsplit) {
406:    if (!params.skip_alignment && params.aligner == 'star_salmon') {
509:        if 

**This shows how `--skip_*` parameters control execution paths!**

---
## Part 7: Work Directory Exploration

### 15. Compare Work Directory Sizes

In [ ]:
# See how much space the work directory uses
!du -sh work/

### 16. Find Specific Process Outputs

In [ ]:
# Find all STAR alignment work directories
!nextflow log -f "hash,name" | grep "STAR_ALIGN" | head -5

In [ ]:
# Pick one hash and explore its directory
# Replace <hash> with actual hash from above
# !ls -lh work/<first_two_chars>/<hash>/

---
## Part 8: Resume Feature with Complex Workflows

### 17. Test Resume on Complex Pipeline

In [ ]:
%%bash

# Re-run the full pipeline with resume
# Should complete in seconds using cached results
nextflow run nf-core/rnaseq \
    -profile test,conda \
    --outdir results_rnaseq_full \
    -resume

**Notice:** All 30+ processes were cached!  
In a real run, this saves hours of computation.

---
## Part 9: Reports Analysis

### 18. Compare Full vs Salmon-only Reports

In [ ]:
# List all generated reports
!ls -lh *.html *.txt

### Display Reports in Notebook

The HTML reports contain interactive JavaScript and are best viewed in a browser.  
**Download and open these files locally:**

In [ ]:
# Provide download links for HTML reports
from IPython.display import FileLink, display, Markdown

print("📊 Interactive HTML Reports:")
print("\n1. Execution Report (CPU, memory, duration per process):")
display(FileLink('report_full.html'))

print("\n2. Timeline (Gantt chart of execution):")
display(FileLink('timeline_full.html'))

print("\n3. MultiQC Report (aggregated quality metrics):")
display(FileLink('results_rnaseq_full/multiqc/star_salmon/multiqc_report.html'))

In [ ]:
# Display the MultiQC report
from IPython.display import HTML, display

with open('results_rnaseq_full/multiqc/star_salmon/multiqc_report.html', 'r') as f:
    multiqc_html = f.read()
    
print("📈 MultiQC Report (aggregated QC metrics):")
display(HTML(multiqc_html))

### 19. Analyze Resource Usage

In [23]:
# Find the most resource-intensive processes
!cat trace_full.txt | sort -t $'\t' -k10 -n -r | head -10 | cut -f1,2,10

sort: write failed: 'standard output'36	ef/2277ee	349.8%
: Broken pipe
32	fe/fa343c	339.7%
165	1a/2f3457	336.8%
162	c8/69c8a4	329.0%
sort: 198	f3/fd7964	323.5%
write error
67	7b/935dfc	317.2%
35	de/415259	316.3%
193	65/71c655	309.8%
56	cd/525a3d	301.6%
33	13/40a82f	300.2%


**Column 10 is peak memory usage** - STAR alignment typically uses the most!

---
## Part 10: Advanced Exercises

### 20. Exercise: Compare Different Quantifiers

RNA-seq supports multiple quantification methods. Try running with different ones!

In [24]:
# See available aligner/quantifier options
!nextflow run nf-core/rnaseq --help | grep -A 5 "aligner"

  --aligner                     [string]  Specifies the alignment algorithm to use - available options are 'star_salmon', 
'star_rsem', 'hisat2', and 'bowtie2_salmon'.  (accepted: star_salmon, 
star_rsem, hisat2, bowtie2_salmon) [default: star_salmon]  
  --use_sentieon_star           [boolean] Optionally accelerate STAR with Sentieon 
  --use_parabricks_star         [boolean] Optionally accelerate STAR and MarkDuplicates with Parabricks 
  --pseudo_aligner              [string]  Specifies the pseudo aligner to use - available options are 'salmon'. Runs in 
addition to '--aligner'.  (accepted: salmon, kallisto)  
  --pseudo_aligner_kmer_size    [integer] Kmer length passed to indexing step of pseudoaligners [default: 31] 
  --bam_csi_index               [boolean] Create a CSI index for BAM files instead of the traditional BAI index. This will 
be required for genomes with larger chromosome sizes.  
  --star_ignore_sjdbgtf         [boolean] When using pre-built STAR indices do not re-ex

**Try running with:**
```bash
# RSEM quantification
nextflow run nf-core/rnaseq \
    -profile test,conda \
    --aligner star_rsem \
    --outdir results_rsem \
    -resume

# kallisto pseudo-alignment
nextflow run nf-core/rnaseq \
    -profile test,conda \
    --pseudo_aligner kallisto \
    --skip_alignment \
    --outdir results_kallisto \
    -resume
```

---
## Key Takeaways: Demo vs RNA-seq

### What Demo Taught:
- ✅ Basic process definition
- ✅ Simple channels
- ✅ Linear workflow
- ✅ Output publishing

### What RNA-seq Adds:
- ✨ **Subworkflows** - Composable workflow sections
- ✨ **Conditional execution** - Branching logic with skip parameters
- ✨ **Module organization** - How to structure large pipelines
- ✨ **Tool alternatives** - Running different implementations
- ✨ **Complex DAGs** - Understanding non-linear workflows
- ✨ **Channel merging** - Collecting outputs from multiple branches
- ✨ **Production patterns** - Error handling, resource management

---

## Next Steps

### 🎯 Immediate Exercises

1. **Study the ALIGN_STAR subworkflow**
   ```bash
   cat ~/.nextflow/assets/nf-core/rnaseq/subworkflows/local/align_star.nf
   ```
   Understand how it composes multiple modules

2. **Trace a sample through the workflow**
   - Find where input is validated
   - Follow it through FastQC
   - See it enter alignment
   - Watch quantification happen
   - See results merge to MultiQC

3. **Modify a skip parameter**
   - Run with `--skip_fastqc`
   - Compare the DAG
   - See how `-resume` handles it

4. **Compare trace files**
   - Full run vs Salmon-only
   - Which processes were skipped?
   - How much time was saved?

### 🚀 Next Level

Now you're ready to **build your own pipeline** using nf-core patterns:

1. **Structure** - Main workflow + subworkflows
2. **Modules** - Reuse nf-core modules or create your own
3. **Conditionals** - Add skip parameters
4. **Testing** - Create test profiles

### 📚 Study These Files

```bash
# Main workflow entry
~/.nextflow/assets/nf-core/rnaseq/main.nf

# Main workflow logic
~/.nextflow/assets/nf-core/rnaseq/workflows/rnaseq.nf

# Subworkflows
~/.nextflow/assets/nf-core/rnaseq/subworkflows/local/

# Modules
~/.nextflow/assets/nf-core/rnaseq/modules/nf-core/

# Configuration
~/.nextflow/assets/nf-core/rnaseq/nextflow.config
```

---

## 🎓 You've Now Mastered

- ✅ Complex workflow structures
- ✅ Subworkflow composition
- ✅ Conditional execution
- ✅ Module organization
- ✅ Production-grade patterns
- ✅ Tool alternatives and flexibility
- ✅ DAG visualization and analysis
- ✅ Resource optimization with skip flags

**You're ready to build real pipelines! 🎉**

---

## 💡 Pro Tips for Building Your Own

1. **Start simple** - One process, get it working, then add more
2. **Copy modules** - Don't reinvent, adapt existing nf-core modules
3. **Test locally** - Use `-profile conda` before moving to AWS Batch
4. **Use subworkflows** - Group related processes (like RNA-seq does)
5. **Add skip flags** - Make your pipeline flexible from day one
6. **Document with DAGs** - Always generate workflow visualizations
7. **Create test profiles** - Small test data for quick iteration

---

**Happy Advanced Nextflow Learning! 🚀**